In [1]:
from openai import OpenAI
openai_client = OpenAI()

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from rag_helper import RAGBase

Чтение файлов

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1

In [6]:
len(documents)

72

Q2

In [7]:
from minsearch import Index

In [8]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [9]:
index = build_index(documents)

In [10]:
index.search("How does the agentic loop keep calling the model until it stops?")[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3

In [11]:
rag_base = RAGBase(index=index, llm_client=openai_client)

In [12]:
response,answer = rag_base.rag("How does the agentic loop keep calling the model until it stops?")

In [13]:
response.usage.input_tokens

7126

Q4

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
print(len(chunks))

295


Q5

In [16]:
chunks_index = build_index(chunks)

In [17]:
chunks_rag_base = RAGBase(index=chunks_index, llm_client=openai_client)

In [18]:
chunks_response,chunks_answer = chunks_rag_base.rag("How does the agentic loop keep calling the model until it stops?")

In [19]:
response.usage.input_tokens // chunks_response.usage.input_tokens

3

Q6

In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
agent_tools = Tools()

In [22]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict = {'content': 3.0, 'filename': 0.5}
    )

In [23]:
agent_tools.add_tool(search)

In [24]:
AGENT_INSTRUCTIONS = """You're a course teaching assistant. Answer the student's question using the search tool.
 Make multiple searches with different keywords before answering."""

In [25]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [26]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Plain RAG,Agentic loop
Fixed pipeline,Iterative loop
Developer controls steps,LLM controls steps
Usually one search,Can do many searches
"Faster, cheaper, simpler","More flexible, but slower and costlier"
No recovery if search fails,Can retry and adapt
